# 第 8 章 不均一分散: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/econometrics-programming/08-heteroskedasticity.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0                # 乱数の種
n = 100                 # 標本の大きさ
heteroskedastic = True  # True: x が大きいほど誤差が大きい
repetitions = 1000      # 標準誤差を比べる繰り返しの回数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 検定と、頑健な標準誤差

In [ ]:
import numpy as np

from lib.chapter03.multiple import fit_ols
from lib.chapter08.hetero import (
    breusch_pagan,
    draw_fan_sample,
    reset_test,
    robust_standard_errors,
    standard_error_comparison,
    white_test,
)

x, y = draw_fan_sample(n, np.random.default_rng(seed), heteroskedastic)
result = fit_ols(y, x)
print(f"通常の標準誤差 {result.standard_errors.iloc[1]:.3f}、"
      f"頑健な標準誤差（HC1） {robust_standard_errors(y, x).iloc[1]:.3f}")
print(f"Breusch-Pagan の p 値 {breusch_pagan(y, x).p_value:.4f}、"
      f"White の p 値 {white_test(y, x).p_value:.4f}、"
      f"RESET の p 値 {reset_test(y, x).p_value:.4f}")

rng = np.random.default_rng(seed)
r = standard_error_comparison(n, repetitions, heteroskedastic, rng)
print(f"傾きの実際のばらつき {r['sd']:.3f}、通常 {r['classical']:.3f}、"
      f"頑健 {r['hc1']:.3f}")

## 図: 実際のばらつきと標準誤差

In [ ]:
from lib.chapter08.plot import plot_standard_errors

plot_standard_errors(
    {
        "Actual (SD of estimates)": r["sd"],
        "Classical": r["classical"],
        "Robust (HC1)": r["hc1"],
    }
)

## 試してみよう

- `heteroskedastic` を `False` にすると、通常の標準誤差（0.081）
  と頑健な標準誤差（0.082）は、ほぼ同じになり、Breusch-Pagan の p 値は 0.4135 で、
  均一分散を棄却しません
- `True` のとき、実際のばらつき 0.200 に対して、通常の標準誤差は 0.157 と小さすぎ、
  頑健な標準誤差は 0.193 で近くなります（図の 3 本の棒）